# Conditional Routing Agent Workflow — Selecting the Right Model for Each Task

This notebook demonstrates how to build a Conditional Routing Agent that automatically selects the most suitable model for a given task using Clarifai’s OpenAI-compatible API.

The workflow routes incoming queries to different models based on their strengths. For example, a coding prompt can be handled by [Qwen3-Coder-30B](https://clarifai.com/qwen/qwenCoder/models/Qwen3-Coder-30B-A3B-Instruct), a reasoning-heavy question can go to [GPT-OSS-120B](https://clarifai.com/openai/chat-completion/models/gpt-oss-120b), and conversational or summarization queries can use [Llama-3.2-3B-Instruct](https://clarifai.com/meta/Llama-3/models/Llama-3_2-3B-Instruct).

The routing process is handled by a lightweight Router LLM, which evaluates the input and returns a structured JSON response specifying the model to use and the reasoning behind its choice. Once the model is selected, the workflow automatically executes the query and returns the result.

Each step of this notebook is modular, from defining the model catalog to running complete routing and execution examples. The implementation highlights how multi-model orchestration can be achieved cleanly and efficiently using Clarifai’s OpenAI-compatible API.

# Setup and Configuration

To get started, install the openai package for API calls and pydantic for schema validation.

Set up your [Clarifai Personal Access Token (PAT)](https://docs.clarifai.com/control/authentication/pat/#how-to-create-a-pat-on-the-platform) from Settings → Secrets in the Clarifai Portal, and initialize the OpenAI client to use Clarifai’s OpenAI-compatible endpoint.

In [1]:
# Install required packages

!pip install -q openai pydantic

In [ ]:
# Imports and Configuration

import os
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Dict, Type

# Initialize the OpenAI client using Clarifai's API endpoint
client = OpenAI(
    base_url="https://api.clarifai.com/v2/ext/openai/v1",  # Clarifai's OpenAI-compatible endpoint
    api_key=os.environ["CLARIFAI_PAT"]                     # Uses your Clarifai PAT stored as an environment variable
)

In [5]:
# Define Model Catalog

MODEL_CATALOG = {
    "gpt_oss_120b": {
        "url": "https://clarifai.com/openai/chat-completion/models/gpt-oss-120b",
        "description": "Advanced open-source model optimized for reasoning, structured generation, and complex analytical tasks.",
        "best_for": "Reasoning, text generation, summarization, and general question answering."
    },
    "qwen3_coder_30b": {
        "url": "https://clarifai.com/qwen/qwenCoder/models/Qwen3-Coder-30B-A3B-Instruct",
        "description": "High-performance coding model designed for long-context reasoning and program synthesis.",
        "best_for": "Code generation, debugging, algorithm design, and development workflows."
    },
    "llama3_2_3b_instruct": {
        "url": "https://clarifai.com/meta/Llama-3/models/llama-3_2-3B-Instruct",
        "description": "Instruction-tuned conversational model with strong multilingual and summarization capabilities.",
        "best_for": "Dialogue, summarization, multilingual assistance, and general-purpose chat tasks."
    },
    "gemma3_12b_it": {
        "url": "https://clarifai.com/gcp/generate/models/gemma-3-12b-it",
        "description": "Multimodal and multilingual model tuned for instruction following and long-context understanding.",
        "best_for": "Image and text reasoning, content generation, summarization, and multilingual tasks."
    }
}

# Display available models
print("Available Models:")
print("=" * 80)
for name, info in MODEL_CATALOG.items():
    print(f"\n{name}:")
    print(f"  Best for: {info['best_for']}")

Available Models:

gpt_oss_120b:
  Best for: Reasoning, text generation, summarization, and general question answering.

qwen3_coder_30b:
  Best for: Code generation, debugging, algorithm design, and development workflows.

llama3_2_3b_instruct:
  Best for: Dialogue, summarization, multilingual assistance, and general-purpose chat tasks.

gemma3_12b_it:
  Best for: Image and text reasoning, content generation, summarization, and multilingual tasks.


In [6]:
# Define Routing Schema

class RouteSelection(BaseModel):
    """Schema for the router's model selection decision."""
    selected_model: str = Field(
        description="Key of the selected model from the catalog."
    )
    justification: str = Field(
        description="Brief explanation of why this model was chosen for the task."
    )

print("✓ Route selection schema defined successfully!")


✓ Route selection schema defined successfully!


In [7]:
# Helper Function — Call LLM

def call_model(prompt: str, model_url: str, image_url: str = None) -> str:
    """
    Helper to call a Clarifai-hosted model via the OpenAI-compatible API.
    
    Args:
        prompt: User input or task description.
        model_url: URL of the Clarifai model to use.
        image_url: Optional image URL for multimodal inputs.
        
    Returns:
        The model's generated text response.
    """
    print(f"\nCalling model: {model_url}")
    print("------------------------------------------------------------")
    
    if image_url:
        print(f"Image input detected: {image_url}")
        response = client.chat.completions.create(
            model=model_url,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": image_url}}
                    ]
                }
            ],
            temperature=0.0,
            max_tokens=2048
        )
    else:
        response = client.chat.completions.create(
            model=model_url,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=1024,
            temperature=0.7,
        )

    result = response.choices[0].message.content

    return result

print("Model calling helper ready.")

Model calling helper ready.


In [ ]:
# Router LLM with Structured Outputs

def call_router_llm(prompt: str, schema: Type[BaseModel]) -> Dict:
    """
    Calls the router LLM to get a structured JSON decision using Structured Outputs.

    Args:
        prompt: The routing decision prompt.
        schema: The expected output schema (Pydantic BaseModel).

    Returns:
        Parsed JSON dictionary conforming to the provided schema.
    """
    router_model_url = "https://clarifai.com/moonshotai/kimi/models/Kimi-K2-Instruct"

    response = client.chat.completions.create(
        model=router_model_url,
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema.__name__,
                "schema": schema.model_json_schema(),
                "strict": True
            }
        },
        temperature=0.3,  # Lower temperature for deterministic JSON
        max_completion_tokens=512
    )
    # Get the text output from the model
    response_text = response.choices[0].message.content.strip()

    # Parse JSON into a Python dictionary
    try:
        parsed_output = json.loads(response_text)
        return parsed_output
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse router response as JSON: {e}")

print("✓ Router LLM helper ready!")


✓ Router LLM helper ready!


In [11]:
# Build Routing Prompt

def build_routing_prompt(user_query: str) -> str:
    """
    Build the routing prompt for selecting the most suitable model.
    
    Args:
        user_query: The user's task or query.
    
    Returns:
        Formatted routing prompt string.
    """
    model_info = "\n".join([
        f"- {name}: {info['best_for']}"
        for name, info in MODEL_CATALOG.items()
    ])
    
    prompt = f"""
                You are a model routing system. Given a user query, select the most appropriate model from the list below.

                Available Models:
                {model_info}

                User Query:
                {user_query}

                Respond ONLY in valid JSON format:
                {{
                    "selected_model": "<model_key_from_catalog>",
                    "justification": "<brief reason why this model is best suited>"
                }}
            """
    return prompt

print("✓ Router prompt builder ready!")


✓ Router prompt builder ready!


In [12]:
# Main Routing Workflow

def route_and_execute(user_query: str, image_url: str = None) -> Dict:
    """
    Complete routing workflow:
    1. Build routing prompt.
    2. Get routing decision from router model.
    3. Execute task with the selected model (supports text and multimodal inputs).
    
    Args:
        user_query: The user's task or question.
        image_url: Optional image URL for multimodal inputs.
    
    Returns:
        Dictionary with selected model, justification, and response.
    """

    print("\n[Step 1] Building routing prompt...")
    routing_prompt = build_routing_prompt(user_query)
    print("Routing prompt created successfully.")
    print("------------------------------------------------------------\n")

    print("[Step 2] Calling router model to analyze and select the best model...")
    routing_decision = call_router_llm(routing_prompt, RouteSelection)

    selected_model = routing_decision["selected_model"]
    justification = routing_decision["justification"]

    print("\nRouting Decision:")
    print(f"  Selected Model: {selected_model}")
    print(f"  Justification: {justification}")
    print("============================================================\n")

    print("[Step 3] Executing the task with the selected model...")
    model_url = MODEL_CATALOG[selected_model]["url"]

    # Single call handles both text and multimodal paths
    final_response = call_model(user_query, model_url, image_url=image_url)

    print("[Step 4] Workflow complete.")
    print("============================================================\n")

    return {
        "selected_model": selected_model,
        "justification": justification,
        "response": final_response
    }

In [21]:
## Example 1 — Coding Task

print("EXAMPLE 1: Coding Task")
print("=" * 80 + "\n")

coding_query = (
    "Write a Python function to find all prime numbers up to 100."
)

result_1 = route_and_execute(coding_query)

print("\nModel Response:\n")
print(result_1["response"])
print("\n" + "=" * 80)


EXAMPLE 1: Coding Task


[Step 1] Building routing prompt...
Routing prompt created successfully.
------------------------------------------------------------

[Step 2] Calling router model to analyze and select the best model...

Routing Decision:
  Selected Model: qwen3_coder_30b
  Justification: The request is purely algorithmic Python code generation, which falls squarely under the strengths of the coding-specialized model.

[Step 3] Executing the task with the selected model...

Calling model: https://clarifai.com/qwen/qwenCoder/models/Qwen3-Coder-30B-A3B-Instruct
------------------------------------------------------------
[Step 4] Workflow complete.


Model Response:

Here's a Python function to find all prime numbers up to 100 using the Sieve of Eratosthenes algorithm:

```python
def find_primes_up_to_100():
    """
    Find all prime numbers up to 100 using the Sieve of Eratosthenes algorithm.
    
    Returns:
        list: A list of all prime numbers up to 100
    """
    # 

In [106]:
## Example 2 — Multimodal Task (Image + Text)

print("EXAMPLE 2: Multimodal Task (Image + Text)")
print("=" * 80 + "\n")

image_url = "https://samples.clarifai.com/cat1.jpeg"
multimodal_query = "Describe the scene in the image and infer what the cat might be doing."

result_2 = route_and_execute(multimodal_query, image_url=image_url)

print("Selected Model:", result_2["selected_model"])
print("Justification:", result_2["justification"])
print("\nModel Response:\n")
print(result_2["response"])
print("\n" + "=" * 80)

EXAMPLE 2: Multimodal Task (Image + Text)


[Step 1] Building routing prompt...
Routing prompt created successfully.
------------------------------------------------------------

[Step 2] Calling router model to analyze and select the best model...

Routing Decision:
  Selected Model: gemma3_12b_it
  Justification: The request explicitly asks to describe an image and reason about what the cat in it might be doing, so the model must be able to process both image and text inputs; gemma3_12b_it is the only listed model that supports vision reasoning.

[Step 3] Executing the task with the selected model...

Calling model: https://clarifai.com/gcp/generate/models/gemma-3-12b-it
------------------------------------------------------------
Image input detected: https://samples.clarifai.com/cat1.jpeg
[Step 4] Workflow complete.

Selected Model: gemma3_12b_it
Justification: The request explicitly asks to describe an image and reason about what the cat in it might be doing, so the model must be 

In [102]:
## Example 3 — Complex Reasoning Task

print("EXAMPLE 3: Complex Reasoning Task")
print("=" * 80 + "\n")

reasoning_query = """
Four people need to cross a bridge at night with one flashlight.
The bridge can hold only two people at a time.
They take 1, 2, 5, and 10 minutes to cross, respectively.
When two people cross together, they move at the slower person's pace.
What is the minimum total time required for all four people to cross the bridge?
"""

result_3 = route_and_execute(reasoning_query)

print("Selected Model:", result_3["selected_model"])
print("Justification:", result_3["justification"])
print("\nModel Response:\n")
print(result_3["response"])
print("\n" + "=" * 80)

EXAMPLE 3: Complex Reasoning Task


[Step 1] Building routing prompt...
Routing prompt created successfully.
------------------------------------------------------------

[Step 2] Calling router model to analyze and select the best model...

Routing Decision:
  Selected Model: gpt_oss_120b
  Justification: This is a classic logic/optimization puzzle that requires reasoning and step-by-step minimization rather than code generation or multilingual chat.

[Step 3] Executing the task with the selected model...

Calling model: https://clarifai.com/openai/chat-completion/models/gpt-oss-120b
------------------------------------------------------------
[Step 4] Workflow complete.

Selected Model: gpt_oss_120b
Justification: This is a classic logic/optimization puzzle that requires reasoning and step-by-step minimization rather than code generation or multilingual chat.

Model Response:

**Answer: 17 minutes**

---

### Why 17 minutes is optimal  

Let the four people be  

- **A** – 1 min  
- 

In [107]:
## Example 4 — Mixed Batch of Prompts

print("EXAMPLE 4: Mixed Batch of Prompts")
print("=" * 80 + "\n")

prompts = [
    "Write a Python function that finds the longest common prefix among a list of strings.",
    "Summarize this paragraph: Machine learning models improve by learning from data rather than being explicitly programmed.",
    "Describe the main elements and possible context of this image.",
    "Write a short story about a robot who learns to paint.",
    "Plan a 3-day trip to Italy focused on history, art, and cuisine."
]

for i, query in enumerate(prompts, start=1):
    print(f"\n--- Task {i} ---")
    print(f"Prompt: {query}")
    print("-" * 80)

    image_input = None
    # Assign an image for the multimodal example
    if "image" in query.lower():
        image_input = "https://samples.clarifai.com/cat1.jpeg"

    result = route_and_execute(query, image_url=image_input)

    print("Selected Model:", result["selected_model"])
    print("Justification:", result["justification"])
    print("\nModel Response:\n")
    print(result["response"])
    print("\n" + "=" * 80)

EXAMPLE 4: Mixed Batch of Prompts


--- Task 1 ---
Prompt: Write a Python function that finds the longest common prefix among a list of strings.
--------------------------------------------------------------------------------

[Step 1] Building routing prompt...
Routing prompt created successfully.
------------------------------------------------------------

[Step 2] Calling router model to analyze and select the best model...

Routing Decision:
  Selected Model: qwen3_coder_30b
  Justification: The request is explicitly a code-generation task—writing a Python function—so the model specialized in coding and algorithm design is the best fit.

[Step 3] Executing the task with the selected model...

Calling model: https://clarifai.com/qwen/qwenCoder/models/Qwen3-Coder-30B-A3B-Instruct
------------------------------------------------------------
[Step 4] Workflow complete.

Selected Model: qwen3_coder_30b
Justification: The request is explicitly a code-generation task—writing a Python fun